In [ ]:
# pip install selenium beautifulsoup4 sentence-transformers chromadb --quiet


# FOR CONCERT

In [ ]:
# Prints all URLs to visit in the ConcertAPI docs - use these to navigate required URLs to fetch information

import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---------- CONFIG ----------
BASE_URL = "https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0"
DOMAIN_PREFIX = "https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/"

# Sidebar navigation selector - the main ul container with all the divs
SIDEBAR_SELECTOR = "#root > div > div.overview-page.hide-scrollbar.grid-width > div.flex-grid.cds--grid > div.custom-grid-1 > div > div.side-nav-column.pr-0.cds--col-sm-4.cds--col-md-2.cds--col-lg-3.cds--col-xlg-3.cds--col-max-3 > div.side-nav > nav > ul > div:nth-child(3) > li > ul"

# ---------- SETUP ----------
os.makedirs("concert_api_docs/meta", exist_ok=True)

# Initialize Chrome with options
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")  # Use new headless mode
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

visited = set()
all_links = set()

# ---------- HELPERS ----------

def extract_sidebar_links(driver, base_url):
    """Extract all navigation links from the sidebar"""
    links = set()
    
    # Wait for the sidebar to load
    wait = WebDriverWait(driver, 10)
    sidebar = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, SIDEBAR_SELECTOR)))
    
    # Find all div children under the main ul
    divs = sidebar.find_elements(By.XPATH, "./div")
    print(f"Found {len(divs)} main navigation sections")
    
    for i, div in enumerate(divs, 1):
        try:
            # Look for nested ul > li > ul structure within each div
            nested_uls = div.find_elements(By.XPATH, ".//ul")
            for ul in nested_uls:
                # Get all anchor tags within this ul
                anchors = ul.find_elements(By.TAG_NAME, "a")
                for anchor in anchors:
                    href = anchor.get_attribute("href")
                    if href and href.startswith(DOMAIN_PREFIX):
                        links.add(href)
                        print(f"Found link: {href}")
                    elif href and href.startswith("#"):
                        # Handle fragment-only URLs
                        full_url = base_url + href
                        links.add(full_url)
                        print(f"Found fragment link: {full_url}")
        except Exception as e:
            print(f"Error processing div {i}: {e}")
            continue
    
    return list(links)

# ---------- MAIN EXECUTION ----------

try:
    # First, visit the base URL and extract all sidebar links
    print(f"Visiting base URL: {BASE_URL}")
    driver.get(BASE_URL)
    time.sleep(3)  # Wait for page to load completely
    
    # Extract all navigation links from sidebar
    all_links = extract_sidebar_links(driver, BASE_URL)
    print(f"\nFound {len(all_links)} total links")
    
except Exception as e:
    print(f"[CRITICAL ERROR]: {e}")

finally:
    driver.quit()

Visiting base URL: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0
Found 31 main navigation sections
Found link: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#post_login
Found link: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#delete_apikey
Found link: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#get_apikey
Found link: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#post_apikey
Found link: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#list_users_and_groups
Found link: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#list_all_applications
Found link: https://developer.ibm.com/apis/catal

In [ ]:
# # Navigates all the URLs and fetches required pages. Info is stored into ChromaDB

# import os
# import time
# import hashlib
# import re
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from sentence_transformers import SentenceTransformer
# import chromadb
# from chromadb.config import Settings
# from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


# # ---------- CONFIG ----------
# prefix = "https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#"
# suffixes = [
#     "post_login",
#     "delete_apikey",
#     "get_apikey",
#     "post_apikey",
#     "list_users_and_groups",
#     "list_all_applications",
#     "create_applications",
#     "delete_application",
#     "get_application_details",
#     "update_applications",
#     "list_applications_vulnerability_details",
#     "get_application_package_trends",
#     "list_environment_certificates",
#     "get_certificate_details",
#     "get_certificate_issuers",
#     "update_certificates_data",
#     "delete_certificates_data",
#     "list_postures",
#     "get_posture_by_id",
#     "create_postures_update",
#     "create_posture",
#     "list_catalogs",
#     "create_catalog",
#     "delete_catalog",
#     "get_catalog_by_id",
#     "import_catalogs",
#     "list_all_profiles",
#     "create_profile",
#     "delete_profile",
#     "get_profile_by_id",
#     "replace_profile",
#     "import_profiles",
#     "get_all_risks",
#     "upsert_risk",
#     "delete_risk",
#     "create_delete_component_definitions",
#     "import_component_definitions",
#     "upload_component_definitions",
#     "list_concert_workflows",
#     "create_components",
#     "list_concertdef_sboms",
#     "validate_concertdef_sbom",
#     "concertdef_sbom_schema",
#     "delete_credentials",
#     "list_credentials",
#     "create_credential",
#     "get_credential",
#     "update_credential",
#     "validate_credential",
#     "delete_entity_access_users",
#     "list_entity_access_users",
#     "update_entity_access_users",
#     "post_entity_access_users",
#     "delete_entity_settings",
#     "get_entity_settings",
#     "update_entity_settings",
#     "create_entity_settings",
#     "list_environments",
#     "create_environment",
#     "delete_environment",
#     "get_environment",
#     "update_environment",
#     "list_runtimes",
#     "get_runtime",
#     "get_el_object",
#     "add_el_object",
#     "create_applications_export",
#     "download_export_jobs",
#     "create_environments_export",
#     "export_el_object",
#     "export_el_files",
#     "create_export_catalog",
#     "create_export_profile",
#     "create_export_assessment",
#     "delete_handler_open",
#     "get_one_handler_open",
#     "create_handler_open",
#     "replace_handler_description_open",
#     "replace_handler_protocol_open",
#     "replace_handler_hostname_open",
#     "replace_handler_port_open",
#     "replace_handler_prefix_open",
#     "replace_handler_auth_open",
#     "replace_handler_default_headers_open",
#     "list_handlers_open",
#     "delete_contract_open",
#     "get_contract_information_open",
#     "create_contract_open",
#     "replace_contract_open",
#     "ingest_upload_files",
#     "ingest_load_data",
#     "import_files",
#     "list_ingestion_jobs",
#     "create_ingestion_job",
#     "delete_ingestion_job",
#     "get_ingestion_job",
#     "update_ingestion_job",
#     "collect_data",
#     "list_images",
#     "list_repositories",
#     "list_instance_settings",
#     "update_instance_setting",
#     "list_build_artifacts",
#     "create_build_artifacts",
#     "delete_build_artifact",
#     "get_artifact",
#     "update_build_artifact",
#     "get_artifact_packages",
#     "get_artifact_cves",
#     "get_artifact_blast_radius",
#     "list_build_artifacts_vulnerability_details",
#     "list_build_artifacts_history",
#     "list_build_artifact_package_trends",
#     "list_source_repos",
#     "create_source_repos",
#     "delete_source_repos",
#     "get_source_repository_details",
#     "update_source_repos",
#     "get_source_repo_blast_radius",
#     "list_sourcerepo_history",
#     "list_sourcerepo_package_trends",
#     "list_accesspoints",
#     "get_accesspoint_details",
#     "get_accesspoint_blast_radius",
#     "create_manual_assessments",
#     "create_manual_assessment_update",
#     "create_job",
#     "get_job_by_id",
#     "update_cancel_job",
#     "list_packages",
#     "list_packages_vulnerability_details",
#     "list_recommendations",
#     "get_recommendation",
#     "get_sca_package_highlights",
#     "get_sca_sbom_highlights",
#     "get_sca_package",
#     "list_cyclonedx_sboms",
#     "delete_sbom",
#     "get_cyclonedx_sbom",
#     "get_sbom_blast_radius",
#     "get_sbom_openssf_scorecard",
#     "get_package_blast_radius",
#     "get_package_associations",
#     "list_sbom_history",
#     "get_tocs_lz_filters",
#     "search_tocs_lz",
#     "highlight_tocs_lz",
#     "search_tocs_el",
#     "upload_evidence",
#     "archive_record",
#     "list_all_tags",
#     "create_tags",
#     "delete_tag",
#     "update_tags",
#     "get_user_preferences",
#     "replace_user_preferences",
#     "get_table_preferences",
#     "replace_table_preferences",
#     "list_vulnerabilities",
#     "get_vulnerability",
#     "update_vulnerabilities",
#     "list_vulnerability_scans",
#     "list_package_vulnerabilities",
#     "list_dast_and_sast_exposures",
#     "get_vulnerability_blast_radius",
#     "get_exposure",
#     "list_vulnerability_assessments",
#     "update_vulnerability_assessment",
#     "get_vulnerability_assessment_by_id",
#     "update_cve_assessment",
#     "list_all_vulnerability_assessments",
#     "delete_vulnerabilities",
#     "delete_exposures",
#     "create_remediation_action",
#     "update_decision_remediation_action",
#     "update_remediation_action",
#     "list_remediation_actions",
#     "get_remediation_action",
#     "search_remediation_actions",
#     "update_exposure_assessment",
#     "get_exposure_trends",
#     "get_vulnerability_trends",
#     "delete_automations",
#     "list_automations",
#     "create_automation",
#     "get_automation",
#     "update_automation_rule",
#     "ticket_create",
#     "get_labels",
#     "triggerEvaluation",
#     "refreshPackageData"
# ]

# # ["post_login", "delete_apikey", "get_apikey"]  # Add all desired suffixes

# DEBUG = True # Set this to True to only print chunks, False to do full processing (storing into ChromaDB)

# chroma_path = "concert_docs/chroma_db"
# collection_name = "concert_api"
# embedding_fn = SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")

# # ---------- SETUP ----------
# if not DEBUG:
#     client = chromadb.PersistentClient(path=chroma_path)
#     collection = client.get_or_create_collection(
#         name=collection_name,
#         embedding_function=embedding_fn
#     )

# # Initialize Chrome with options
# options = webdriver.ChromeOptions()
# options.add_argument("--headless=new")  # Use new headless mode
# options.add_argument("--no-sandbox")
# options.add_argument("--disable-dev-shm-usage")
# options.add_argument("--disable-gpu")
# options.add_argument("--window-size=1920,1080")
# driver = webdriver.Chrome(options=options)

# # ---------- HELPERS ----------

# def accept_cookies(driver):
#     """Accept cookies if banner appears."""
#     try:
#         cookie_button = WebDriverWait(driver, 5).until(
#             EC.element_to_be_clickable((By.CSS_SELECTOR, "#truste-consent-button"))
#         )
#         cookie_button.click()
#         print("Accepted cookies")
#         time.sleep(1)
#     except Exception as e:
#         print(f"No cookie banner or couldn't accept: {e}")

# def wait_for_main_content(driver):
#     """Wait for the main content container to load using XPath."""
#     try:
#         WebDriverWait(driver, 15).until(
#             EC.presence_of_element_located((By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div"))
#         )
#         time.sleep(2)  # Additional wait for stability
#         return True
#     except Exception as e:
#         print(f"Timeout waiting for main content: {e}")
#         return False

# def extract_and_clean_text(driver):
#     """Extract all text content and remove HTML tags."""
#     try:
#         # Find the main container using XPath
#         container = driver.find_element(By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div")
#         html_content = container.get_attribute("innerHTML")
        
#         # Remove all HTML tags using regex
#         clean_text = re.sub(r'<[^>]+>', ' ', html_content)
        
#         # Collapse multiple spaces and trim
#         clean_text = ' '.join(clean_text.split()).strip()
        
#         return clean_text
#     except Exception as e:
#         print(f"Failed to extract and clean content: {e}")
#         return None

# # ---------- MAIN PROCESSING ----------

# for i, suffix in enumerate(suffixes):
#     url = prefix + suffix
#     try:
#         print(f"\nProcessing: {url}")
        
#         # Forcefully break context so Selenium treats this as a new navigation
#         driver.get("about:blank")
#         driver.get(url)
#         time.sleep(1.5)
        
#         # Handle cookies
#         # accept_cookies(driver)
        
#         # Wait for main content
#         if not wait_for_main_content(driver):
#             print(f"Main content not loaded for {url}")
#             continue
        
#         # Extract and clean text
#         clean_text = extract_and_clean_text(driver)
#         if not clean_text:
#             print(f"No content extracted for {url}")
#             continue
        
#         if DEBUG:
#             print(f"\n--- Clean Text for {suffix} ---\n{clean_text}...\n")  # Print first 1000 chars
#         else:
#             chunk_id = f"{i}_{suffix}"
#             collection.upsert(
#                 documents=[clean_text],
#                 ids=[chunk_id],
#                 metadatas=[{
#                     "source": url,
#                     "endpoint": suffix
#                 }]
#             )

#     except Exception as e:
#         print(f"[ERROR] {url}: {e}")

# # Clean up
# driver.quit()
# if not DEBUG:
#     print(f"\n✅ Indexed {len(suffixes)} API endpoints into ChromaDB.")
# else:
#     print(f"\nDEBUG MODE: Processed {len(suffixes)} API endpoints")

/home/gunjan/TheToolkit/schema-mapping-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#post_login

--- Clean Text for post_login ---
API reference Expand Generate an auth token (VM only) POST /core/api/v1/login Generates an authentication token to login to a Concert instance deployed on a virtual machine. Request URL {server}/core/api/v1/login Copy to clipboard Security http bearer A valid bearer type and Concert API key required for authentication. JWT Parameters Name Description InstanceId REQUIRED string (header) Unique ID of your Concert instance Pattern: ^[a-zA-Z0-9\-]+$ Minimum length: 0 Maximum length: 1000 Content-type string (header) application/json Accept string (header) application/json Request body Requestbody Try this API Responses 200 successful login 400 bad request, bad data 401 Unauthorized 404 Not Found 409 conflict...


Processing: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#

KeyboardInterrupt: 

In [1]:
import os
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# ---------- CONFIG ----------
prefix = "https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#"
# suffixes = ["post_login", "delete_apikey", "get_apikey"]

suffixes = [
    "post_login",
    "delete_apikey",
    "get_apikey",
    "post_apikey",
    "list_users_and_groups",
    "list_all_applications",
    "create_applications",
    "delete_application",
    "get_application_details",
    "update_applications",
    "list_applications_vulnerability_details",
    "get_application_package_trends",
    "list_environment_certificates",
    "get_certificate_details",
    "get_certificate_issuers",
    "update_certificates_data",
    "delete_certificates_data",
    "list_postures",
    "get_posture_by_id",
    "create_postures_update",
    "create_posture",
    "list_catalogs",
    "create_catalog",
    "delete_catalog",
    "get_catalog_by_id",
    "import_catalogs",
    "list_all_profiles",
    "create_profile",
    "delete_profile",
    "get_profile_by_id",
    "replace_profile",
    "import_profiles",
    "get_all_risks",
    "upsert_risk",
    "delete_risk",
    "create_delete_component_definitions",
    "import_component_definitions",
    "upload_component_definitions",
    "list_concert_workflows",
    "create_components",
    "list_concertdef_sboms",
    "validate_concertdef_sbom",
    "concertdef_sbom_schema",
    "delete_credentials",
    "list_credentials",
    "create_credential",
    "get_credential",
    "update_credential",
    "validate_credential",
    "delete_entity_access_users",
    "list_entity_access_users",
    "update_entity_access_users",
    "post_entity_access_users",
    "delete_entity_settings",
    "get_entity_settings",
    "update_entity_settings",
    "create_entity_settings",
    "list_environments",
    "create_environment",
    "delete_environment",
    "get_environment",
    "update_environment",
    "list_runtimes",
    "get_runtime",
    "get_el_object",
    "add_el_object",
    "create_applications_export",
    "download_export_jobs",
    "create_environments_export",
    "export_el_object",
    "export_el_files",
    "create_export_catalog",
    "create_export_profile",
    "create_export_assessment",
    "delete_handler_open",
    "get_one_handler_open",
    "create_handler_open",
    "replace_handler_description_open",
    "replace_handler_protocol_open",
    "replace_handler_hostname_open",
    "replace_handler_port_open",
    "replace_handler_prefix_open",
    "replace_handler_auth_open",
    "replace_handler_default_headers_open",
    "list_handlers_open",
    "delete_contract_open",
    "get_contract_information_open",
    "create_contract_open",
    "replace_contract_open",
    "ingest_upload_files",
    "ingest_load_data",
    "import_files",
    "list_ingestion_jobs",
    "create_ingestion_job",
    "delete_ingestion_job",
    "get_ingestion_job",
    "update_ingestion_job",
    "collect_data",
    "list_images",
    "list_repositories",
    "list_instance_settings",
    "update_instance_setting",
    "list_build_artifacts",
    "create_build_artifacts",
    "delete_build_artifact",
    "get_artifact",
    "update_build_artifact",
    "get_artifact_packages",
    "get_artifact_cves",
    "get_artifact_blast_radius",
    "list_build_artifacts_vulnerability_details",
    "list_build_artifacts_history",
    "list_build_artifact_package_trends",
    "list_source_repos",
    "create_source_repos",
    "delete_source_repos",
    "get_source_repository_details",
    "update_source_repos",
    "get_source_repo_blast_radius",
    "list_sourcerepo_history",
    "list_sourcerepo_package_trends",
    "list_accesspoints",
    "get_accesspoint_details",
    "get_accesspoint_blast_radius",
    "create_manual_assessments",
    "create_manual_assessment_update",
    "create_job",
    "get_job_by_id",
    "update_cancel_job",
    "list_packages",
    "list_packages_vulnerability_details",
    "list_recommendations",
    "get_recommendation",
    "get_sca_package_highlights",
    "get_sca_sbom_highlights",
    "get_sca_package",
    "list_cyclonedx_sboms",
    "delete_sbom",
    "get_cyclonedx_sbom",
    "get_sbom_blast_radius",
    "get_sbom_openssf_scorecard",
    "get_package_blast_radius",
    "get_package_associations",
    "list_sbom_history",
    "get_tocs_lz_filters",
    "search_tocs_lz",
    "highlight_tocs_lz",
    "search_tocs_el",
    "upload_evidence",
    "archive_record",
    "list_all_tags",
    "create_tags",
    "delete_tag",
    "update_tags",
    "get_user_preferences",
    "replace_user_preferences",
    "get_table_preferences",
    "replace_table_preferences",
    "list_vulnerabilities",
    "get_vulnerability",
    "update_vulnerabilities",
    "list_vulnerability_scans",
    "list_package_vulnerabilities",
    "list_dast_and_sast_exposures",
    "get_vulnerability_blast_radius",
    "get_exposure",
    "list_vulnerability_assessments",
    "update_vulnerability_assessment",
    "get_vulnerability_assessment_by_id",
    "update_cve_assessment",
    "list_all_vulnerability_assessments",
    "delete_vulnerabilities",
    "delete_exposures",
    "create_remediation_action",
    "update_decision_remediation_action",
    "update_remediation_action",
    "list_remediation_actions",
    "get_remediation_action",
    "search_remediation_actions",
    "update_exposure_assessment",
    "get_exposure_trends",
    "get_vulnerability_trends",
    "delete_automations",
    "list_automations",
    "create_automation",
    "get_automation",
    "update_automation_rule",
    "ticket_create",
    "get_labels",
    "triggerEvaluation",
    "refreshPackageData"
]

DEBUG = False  # Set to False to store in ChromaDB

chroma_path = "/data/wikidata/chromadb" # "concert_docs/chroma_db" 
collection_name = "concert_api"
embedding_fn = SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")

# ---------- SETUP ----------
if not DEBUG:
    client = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_fn
    )

# Initialize Chrome
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

def wait_for_main_content(driver):
    """Wait for the main content container to load."""
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div"))
        )
        time.sleep(2)
        return True
    except Exception:
        return False

def extract_main_content(driver):
    """Extract main page content."""
    try:
        container = driver.find_element(By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div")
        html_content = container.get_attribute("innerHTML")
        clean_text = re.sub(r'<[^>]+>', ' ', html_content)
        clean_text = ' '.join(clean_text.split()).strip()
        return clean_text
    except Exception:
        return None

def find_and_expand_arrows(driver, container, max_depth=10, current_depth=0):
    """Recursively find and expand arrow elements to extract hidden content."""
    if current_depth >= max_depth:
        return []
    
    extracted_contents = []
    
    try:
        arrow_selector = ".//div[contains(@class, 'arrow') and contains(@class, 'arrow-text') and not(contains(@class, 'arrow-down'))]"
        arrows = container.find_elements(By.XPATH, arrow_selector)
        
        for arrow in arrows:
            try:
                current_classes = arrow.get_attribute("class")
                if "arrow-down" in current_classes:
                    continue
                
                arrow_text = arrow.text.strip()
                
                # Find parent element
                try:
                    parent_element = arrow.find_element(By.XPATH, "./ancestor::li[1]")
                except NoSuchElementException:
                    try:
                        parent_element = arrow.find_element(By.XPATH, "./ancestor::div[contains(@class, 'parameter')][1]")
                    except NoSuchElementException:
                        parent_element = arrow.find_element(By.XPATH, "./..")
                
                initial_text_length = len(parent_element.text)
                
                # Click arrow
                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", arrow)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", arrow)
                time.sleep(1.5)
                
                # Check if expanded
                updated_classes = arrow.get_attribute("class")
                if "arrow-down" in updated_classes:
                    updated_text_length = len(parent_element.text)
                    content_growth = updated_text_length - initial_text_length
                    
                    if content_growth > 10:
                        updated_content = parent_element.get_attribute("innerHTML")
                        clean_content = re.sub(r'<[^>]+>', ' ', updated_content)
                        clean_content = ' '.join(clean_content.split()).strip()
                        
                        content_info = {
                            'arrow_text': arrow_text,
                            'content': clean_content
                        }
                        extracted_contents.append(content_info)
                    
                    # Recursively process nested arrows
                    time.sleep(1)
                    nested_contents = find_and_expand_arrows(driver, parent_element, max_depth, current_depth + 1)
                    extracted_contents.extend(nested_contents)
                
            except Exception:
                continue
                
    except Exception:
        pass
    
    return extracted_contents

def find_target_section(driver):
    """Find the specific section containing expandable arrows."""
    try:
        return driver.find_element(By.XPATH, "/html/body/div[1]/div/div[3]/div[2]/div[2]/div/div[2]/div/div/div[1]/div[1]/div[2]/ul")
    except NoSuchElementException:
        return None

# ---------- MAIN PROCESSING ----------

for i, suffix in enumerate(suffixes):
    url = prefix + suffix
    print(f"\nProcessing: {url} ({suffix})")
    
    try:
        # Navigate to page
        driver.get("about:blank")
        driver.get(url)
        time.sleep(2)
        
        if not wait_for_main_content(driver):
            continue
        
        # Extract main content
        main_content = extract_main_content(driver)
        if not main_content:
            continue
        
        # Extract arrow-expanded content
        expanded_content = []
        target_section = find_target_section(driver)
        if target_section:
            expanded_content = find_and_expand_arrows(driver, target_section)
        
        # Combine all content
        combined_content = f"MAIN CONTENT:\n{main_content}\n\n"
        
        if expanded_content:
            combined_content += "EXPANDED CONTENT:\n"
            for content_info in expanded_content:
                combined_content += f"\n--- {content_info['arrow_text']} ---\n"
                combined_content += content_info['content'] + "\n"
        
        # Output extracted content
        print("Extracted content:")
        print(combined_content)
        print("-" * 80)
        
        # Store in ChromaDB or debug
        if DEBUG:
            # Just print for debugging
            pass
        else:
            chunk_id = f"{i}_{suffix}"
            collection.upsert(
                documents=[combined_content],
                ids=[chunk_id],
                metadatas=[{
                    "source": url,
                    "endpoint": suffix
                }]
            )
        
    except Exception as e:
        print(f"Error processing {url}: {e}")

# Clean up
driver.quit()

if not DEBUG:
    print(f"\n✅ Indexed {len(suffixes)} API endpoints into ChromaDB.")
else:
    print(f"\n✅ Debug mode: Processed {len(suffixes)} API endpoints")

/home/gunjan/TheToolkit/schema-mapping-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing: https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#post_login (post_login)
Extracted content:
MAIN CONTENT:
API reference Expand Generate an auth token (VM only) POST /core/api/v1/login Generates an authentication token to login to a Concert instance deployed on a virtual machine. Request URL {server}/core/api/v1/login Copy to clipboard Security http bearer A valid bearer type and Concert API key required for authentication. JWT Parameters Name Description InstanceId REQUIRED string (header) Unique ID of your Concert instance Pattern: ^[a-zA-Z0-9\-]+$ Minimum length: 0 Maximum length: 1000 Content-type string (header) application/json Accept string (header) application/json Request body Requestbody Try this API Responses 200 successful login 400 bad request, bad data 401 Unauthorized 404 Not Found 409 conflict

EXPANDED CONTENT:

--- 200 ---
200 successful login 200 Headers : Name Description Set-Cookie Header cookie Re

In [ ]:
import chromadb
from chromadb.config import Settings

chroma_path = "/data/wikidata/chromadb" 
collection_name = "concert_api"
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_collection(name=collection_name)
collections = client.list_collections()
print("\n🗂 Collections in ChromaDB:")
for col in collections:
    print(f"- {col.name}")

# Load your specific collection (e.g., "turbonomic")
collection = client.get_collection("concert_api") 

# TO PRINT ALL ENTRIES
print(f"\n📦 Collection '{collection.name}' has {collection.count()} documents.\n")
results = collection.get(limit=3)
print(results)

# TO DELETE A SPECIFIC ENTRY
# bad_id = "032e0fd1d8031a86ec7298652bc62e6b"
# # Delete the entry
# collection.delete(ids=[bad_id])
# print(f"✅ Deleted entry with ID: {bad_id}")

# TO DELETE THE COLLECTION
# client.delete_collection(name="concert_api")

[Collection(name=test_collection)]


In [2]:
# TEST THE RETRIEVAL

results = collection.query(
    query_texts=["Token Details"],
    n_results=3,
    include=["documents", "metadatas"]
)

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("\n🔎 Matched Doc:\n", doc)
    print("Metadata:", meta)



🔎 Matched Doc:
 MAIN CONTENT:
API reference Expand Generate an auth token (VM only) POST /core/api/v1/login Generates an authentication token to login to a Concert instance deployed on a virtual machine. Request URL {server}/core/api/v1/login Copy to clipboard Security http bearer A valid bearer type and Concert API key required for authentication. JWT Parameters Name Description InstanceId REQUIRED string (header) Unique ID of your Concert instance Pattern: ^[a-zA-Z0-9\-]+$ Minimum length: 0 Maximum length: 1000 Content-type string (header) application/json Accept string (header) application/json Request body Requestbody Try this API Responses 200 successful login 400 bad request, bad data 401 Unauthorized 404 Not Found 409 conflict

EXPANDED CONTENT:

--- 200 ---
200 successful login 200 Headers : Name Description Set-Cookie Header cookie Response model Properties Example &amp; Schema Name Description object JWT details Example: {"quickstart":true,"instance_id":"0000-0000-0000-0000"

In [ ]:
query = "token details"

results = collection.query(
    query_texts=[query],
    n_results=2,
    include=["documents", "metadatas"]
)

top_chunks = results["documents"][0]
top_sources = results["metadatas"][0]


In [ ]:
import os
import chromadb
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes, DecodingMethods
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai import Credentials
import re
from typing import List, Dict, Optional


class ConcertAPIDescriptionGenerator:
    def __init__(self, chroma_path: str = "/data/wikidata/chromadb", collection_name: str = "concert_api"):
        """Initialize the RAG system for Concert API description generation."""
        self.chroma_path = chroma_path
        self.collection_name = collection_name
        self.client = chromadb.PersistentClient(path=chroma_path)
        self.collection = self.client.get_collection(name=collection_name)
        
        # Load environment variables
        load_dotenv()
        
        # Initialize Watson AI credentials
        self.credentials = Credentials(
            url="https://us-south.ml.cloud.ibm.com",
            api_key=os.environ["WATSONX_APIKEY"]
        )
        self.project_id = os.environ["WATSONX_PROJECT_ID"]
    
    def parse_input(self, input_string: str) -> Dict[str, str]:
        """
        Parse input string in format: productname_typename.fields.fields...
        
        Args:
            input_string: String in format "productname_typename.field1.field2..."
            
        Returns:
            Dict with parsed components
        """
        parts = input_string.split('.')
        
        if not parts:
            raise ValueError("Invalid input format. Expected: productname_typename.fields...")
        
        # Split the first part to get product name and type name
        first_part = parts[0]
        if '_' in first_part:
            product_name, type_name = first_part.rsplit('_', 1)
        else:
            product_name = first_part
            type_name = ""
        
        fields = parts[1:] if len(parts) > 1 else []
        
        return {
            'product_name': product_name,
            'type_name': type_name,
            'fields': fields,
            'full_path': input_string
        }
    
    # def build_search_queries(self, parsed_input: Dict[str, str]) -> List[str]:
    #     """
    #     Build search queries to find relevant documents in ChromaDB.
        
    #     Args:
    #         parsed_input: Parsed input dictionary
            
    #     Returns:
    #         List of search queries
    #     """
    #     queries = []
        
    #     # Primary query with product and type name
    #     primary_query = f"{parsed_input['product_name']} {parsed_input['type_name']}"
    #     queries.append(primary_query)
        
    #     # Query with fields if available
    #     if parsed_input['fields']:
    #         fields_query = f"{parsed_input['product_name']} {parsed_input['type_name']} {' '.join(parsed_input['fields'])}"
    #         queries.append(fields_query)
        
    #     # Just the type name for broader context
    #     if parsed_input['type_name']:
    #         queries.append(parsed_input['type_name'])
        
    #     return queries

    def build_search_queries(self, parsed_input: Dict[str, str]) -> List[str]:
        queries = []

        # Only use type-level context for now
        if parsed_input["type_name"]:
            queries.append(parsed_input["type_name"])
            # queries.append(f"{parsed_input['product_name']} {parsed_input['type_name']}")

        return queries

    
    def retrieve_relevant_docs(self, parsed_input: Dict[str, str], top_k: int = 2) -> List[Dict]:
        """
        Retrieve relevant documents from ChromaDB.
        
        Args:
            parsed_input: Parsed input dictionary
            top_k: Number of documents to retrieve
            
        Returns:
            List of relevant documents
        """
        search_queries = self.build_search_queries(parsed_input)
        all_results = []
        
        for query in search_queries:
            print(f"Using query: {query}")
            try:
                results = self.collection.query(
                    query_texts=[query],
                    n_results=top_k
                )
                
                # Process results
                if results['documents'] and results['documents'][0]:
                    for i, doc in enumerate(results['documents'][0]):
                        metadata = results['metadatas'][0][i] if results['metadatas'] and results['metadatas'][0] else {}
                        distance = results['distances'][0][i] if results['distances'] and results['distances'][0] else 0
                        
                        all_results.append({
                            'document': doc,
                            'metadata': metadata,
                            'distance': distance,
                            'query': query
                        })
            except Exception as e:
                print(f"Error querying ChromaDB with query '{query}': {e}")
                continue
        
        # Remove duplicates and sort by distance (lower is better)
        unique_results = []
        seen_docs = set()
        
        for result in sorted(all_results, key=lambda x: x['distance']):
            doc_hash = hash(result['document'])
            if doc_hash not in seen_docs:
                unique_results.append(result)
                seen_docs.add(doc_hash)
                
            if len(unique_results) >= top_k:
                break
        
        return unique_results[:top_k]
    
    def create_rag_prompt(self, parsed_input: Dict[str, str], relevant_docs: List[Dict]) -> str:
        """
        Create the RAG prompt for the LLM.
        
        Args:
            parsed_input: Parsed input dictionary
            relevant_docs: List of relevant documents
            
        Returns:
            Formatted prompt string
        """
        # Format the retrieved documents
        context_docs = ""
        for i, doc in enumerate(relevant_docs, 1):
            context_docs += f"Document {i}:\n{doc['document']}\n\n"
        
        # Build field path string
        field_path = ""
        if parsed_input['fields']:
            field_path = f"Fields: {' -> '.join(parsed_input['fields'])}"
        
        rag_prompt = f"""You are an expert API documentation assistant. Your task is to generate a **precise and concise description (2-3 lines)** for a specific **field** within an API type, using only the documentation provided.

**Target Field**: `{parsed_input['fields'][-1]}`  
**Type Context**: `{parsed_input['type_name']}`  
**Product**: `{parsed_input['product_name']}`

**Full Path**: `{parsed_input['full_path']}`

**Documentation Context**:
{context_docs}

**Instructions**:
1. Explain the type (`{parsed_input['type_name']}`) in one short sentence first.
2. Then describe what exactly the **field `{parsed_input['fields'][-1]}` represents**, how it behaves, and why it is important.
3. Be specific and factual. Only use the content provided. Avoid general summaries.

**Output Format**:
Write a crisp 2-3 line paragraph describing the field `{parsed_input['fields'][-1]}`. No pointers and headings, just plain description.

Example:
Input: "concert_ApplicationAssociationsInfo.build_artifacts.version"
Output: "Represents the version identifier of a build artifact associated with an application. It is used to specify the exact library version linked in the application's build context."
Begin now:
"""
# ApplicationAssociations describes the collection of components linked to an application. The version field under build_artifacts specifies the exact versiond of a library associated with the application. Helps track and manage the precise artifact instance involved in the application's build.

        # print(rag_prompt)
        return rag_prompt
    
    def get_wxai_response(self, prompt: str, model_id: str = "mistralai/mistral-large", 
                         max_new_tokens: int = 1000, stop_sequences: List[str] = ["\n\n\n", "<|endoftext|>"]) -> str:
        """
        Get response from Watson AI model.
        
        Args:
            prompt: Input prompt
            model_id: Model identifier
            max_new_tokens: Maximum tokens to generate
            stop_sequences: Sequences to stop generation
            
        Returns:
            Generated response
        """
        parameters = {
            GenParams.MIN_NEW_TOKENS: 0,
            GenParams.MAX_NEW_TOKENS: max_new_tokens,
            GenParams.DECODING_METHOD: DecodingMethods.GREEDY,
            GenParams.REPETITION_PENALTY: 1,
            GenParams.STOP_SEQUENCES: stop_sequences
        }

        model = ModelInference(
            model_id=model_id, 
            params=parameters, 
            credentials=self.credentials,
            project_id=self.project_id
        )
        
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = model.generate_text(prompt=[prompt])
                return response[0]
            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}")
                if attempt == max_retries - 1:
                    raise e
                continue
    
    def generate_description(self, input_string: str, top_k: int = 2, 
                           model_id: str = "mistralai/mistral-large") -> Dict[str, any]:
        """
        Main method to generate API description.
        
        Args:
            input_string: Input in format "productname_typename.fields..."
            top_k: Number of documents to retrieve
            model_id: Model to use for generation
            
        Returns:
            Dictionary with results
        """
        try:
            # Parse input
            parsed_input = self.parse_input(input_string)
            print(f"Parsed input: {parsed_input}")
            
            # Retrieve relevant documents
            relevant_docs = self.retrieve_relevant_docs(parsed_input, top_k)
            print(f"Retrieved {len(relevant_docs)} relevant documents")
            
            if not relevant_docs:
                return {
                    'success': False,
                    'error': 'No relevant documents found in ChromaDB',
                    'input': input_string,
                    'parsed_input': parsed_input
                }
            
            # Create RAG prompt
            rag_prompt = self.create_rag_prompt(parsed_input, relevant_docs)
            
            # Generate description
            description = self.get_wxai_response(rag_prompt, model_id)
            
            return {
                'success': True,
                'input': input_string,
                'parsed_input': parsed_input,
                'relevant_docs': relevant_docs,
                'description': description,
                'prompt_used': rag_prompt
            }
            
        except Exception as e:
            return {
                'success': False,
                'error': str(e),
                'input': input_string
            }


def main():
    """Example usage of the Concert API Description Generator."""
    
    # Initialize the generator
    generator = ConcertAPIDescriptionGenerator()
    
    # Example inputs
    test_inputs = [
        "concert_PackageRecommendation.action",
        "concert_PackageDetailsCollection.packages.recommendation.action",
        "concert_PackageRecommendation.risk_type",
        "concert_ApplicationAssociationsInfo.build_artifacts.version"
        ""
    ]
    
    for input_string in test_inputs:
        print(f"\n{'='*60}")
        print(f"Processing: {input_string}")
        print(f"{'='*60}")
        
        result = generator.generate_description(input_string)
        
        if result['success']:
            print("✅ SUCCESS!")
            print(f"\nGenerated Description:")
            print("-" * 40)
            print(result['description'])
            print("-" * 40)
            print(f"Documents used: {len(result['relevant_docs'])}")
            
            # print("\nContext Documents Used:")
            # print("-" * 40)
            # for i, doc in enumerate(result["relevant_docs"], 1):
            #     print(f"[{i}] From: {doc['metadata'].get('endpoint', 'N/A')}")
            #     print(doc["document"])  # limit to 500 characters
            #     print("-" * 40)
            print("\nMetadata of Documents Used:")
            print("-" * 40)
            for i, doc in enumerate(result["relevant_docs"], 1):
                print(f"[{i}] Metadata: {doc['metadata']}")
                print("-" * 40)




if __name__ == "__main__":
    main()


Processing: concert_PackageRecommendation.action
Parsed input: {'product_name': 'concert', 'type_name': 'PackageRecommendation', 'fields': ['action'], 'full_path': 'concert_PackageRecommendation.action'}
Using query: PackageRecommendation
Retrieved 2 relevant documents


✅ SUCCESS!

Generated Description:
----------------------------------------

**Output**:
The `PackageRecommendation` type provides details about recommended actions for software packages. The `action` field specifies the recommended action to be taken for a package, such as "Upgrade." This field is crucial for guiding users on how to address potential issues or improvements related to the package.
----------------------------------------
Documents used: 2

Metadata of Documents Used:
----------------------------------------
[1] Metadata: {'endpoint': 'list_packages', 'source': 'https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#list_packages'}
----------------------------------------
[2] Metadata: {'endpoint': 'refreshPackageData', 'source': 'https://developer.ibm.com/apis/catalog/concert--ibm-concert-api/api/API--concert--ibm-concert-api-v1-1-0#refreshPackageData'}
----------------------------------------

Processing: concert_Packa

# FOR INSTANA